environment delpher

In [ ]:
import os
import json
from lxml import etree
import requests

In [ ]:
file_path = '../../../DATA/europeans-news/alto/urn=ddd_000023667_mpeg21_p002_alto.alto.xml'
filename = os.path.basename(file_path)
file_name = ''.join(os.path.splitext(filename)[0].split('.')[:-1])
file_name = file_name.replace('_', ':')
print(file_name)

##### Load GLiNER

In [ ]:
from gliner import GLiNER

labels = ["person", "location (geographical)", "organization"]

model = GLiNER.from_pretrained("urchade/gliner_multi-v2.1")
model.eval()
print("ok")

In [ ]:
def spans_to_bio(tokens, spans):
    # tokens: list[str]
    # spans: list[{"start": int, "end": int, "label": str}]

    token_dict = {}
    index = 0
    for i, token in enumerate(tokens):
        token_dict[i] = {"token": token, "offset": index, "label": "O"}
        index += len(list(token)) + 1

    # Update labels based on spans
    for span in spans:
        span_start = span["start"]
        span_end = span["end"]
        span_label = span["label"]
        
        first = True
        for token_idx in token_dict:
            token_offset = token_dict[token_idx]["offset"]
            # Check if token offset falls within span range
            if token_offset >= span_start and token_offset < span_end:
                if first:
                    token_dict[token_idx]["label"] = f"B-{span_label[:3].upper()}"
                    first = False
                else:
                    token_dict[token_idx]["label"] = f"I-{span_label[:3].upper()}"
    
    return token_dict

In [ ]:
def split_long_sentences(text):
    """
    Check if any sentence exceeds 350 characters.
    If yes, break it into smaller chunks (<350) and return updated text.
    """
    # Split by periods to get sentences
    sentences = text.split('.')
    result_sentences = []
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        # Check if sentence exceeds 350 characters
        if len(sentence) > 350:
            # Break into chunks
            words = sentence.split()
            current_chunk = []
            current_length = 0
            
            for word in words:
                # If adding next word exceeds 350, save current chunk
                if current_length + len(word) + 1 > 350 and current_chunk:
                    result_sentences.append(" ".join(current_chunk))
                    current_chunk = [word]
                    current_length = len(word)
                else:
                    current_chunk.append(word)
                    current_length += len(word) + 1
            
            # Add remaining words
            if current_chunk:
                result_sentences.append(" ".join(current_chunk))
        else:
            # Sentence is fine, keep as is
            result_sentences.append(sentence)
    
    # Join sentences back with periods
    return ". ".join(result_sentences) + "."

In [ ]:
import re

def chunk_text_by_word_limit(text, word_limit=300):
    """
    Split text into chunks respecting the word limit while not breaking sentences.
    Handles multiple sentence-ending punctuation marks (., !, ?, ;)
    """
    # Split on sentence boundaries: period, exclamation, question mark, semicolon
    # followed by whitespace or end of string
    sentences = re.split(r'(?<=[.!?;])\s+', text.strip())
    
    chunks = []
    current_chunk = []
    current_word_count = 0
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        
        sentence_words = sentence.split()
        sentence_word_count = len(sentence_words)
        
        # If adding this sentence would exceed limit and we have content, start new chunk
        if current_word_count + sentence_word_count > word_limit and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [sentence]
            current_word_count = sentence_word_count
        else:
            current_chunk.append(sentence)
            current_word_count += sentence_word_count
    
    # Add remaining chunk
    if current_chunk:
        chunks.append(' '.join(current_chunk))
    
    return chunks

In [ ]:
words_limit = 300
def create_block_of_text(page_tree):
    ns = {
            "alto": "http://schema.ccs-gmbh.com/ALTO"
        }
    block_of_text = []
    for text_block in page_tree.findall('.//alto:TextBlock', namespaces=ns):
        id = text_block.get("ID")
        text = []
        wc_score = []
        for string in text_block.findall('.//alto:String', namespaces=ns):
            text.append(string.get("CONTENT"))
            wc_score.append(float(string.get("WC")))
        if len(text) > words_limit:
            # divide text into chunks of max 300 words while preserving sentence boundaries
            chunks = chunk_text_by_word_limit(" ".join(text), word_limit=words_limit)
            for chunk in chunks:
                block_of_text.append({
                    "id": id,
                    "text": chunk,
                    "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
                })
        else:
            block_of_text.append({
                "id": id,
                "text": " ".join(text),
                "wc_score": sum(wc_score) / len(wc_score) if wc_score else 0
            })
    return block_of_text

In [ ]:
### Create blocks of text from the XML file

In [ ]:
parser = etree.XMLParser(recover=True)

# read the XML file and create a tree

url= f"http://resolver.kb.nl/resolve?{file_name}"
response = requests.get(url)
if response.status_code != 200:
    print(f"URL: {url} is not valid.")

page_tree = response.content
page_tree = etree.fromstring(page_tree, parser=parser)

if os.path.exists(file_path):
    with open(file_path, 'rb') as f:
        page_tree = etree.parse(f, parser=parser).getroot()
    # print(f"Successfully loaded XML from: {file_path}")
else:
    print(f"File not found: {file_path}")
    print(f"Current working directory: {os.getcwd()}")

blocks = create_block_of_text(page_tree)

#### create block of text from csv

In [ ]:
def create_block_of_text(df):
    block_of_text = []
    df = df.groupby('TEXTBLOCK_ID').agg({'token': ' '.join}).reset_index()
    for index, row in df.iterrows():
        id = row['TEXTBLOCK_ID']
        text = row['token']
        if len(text.split()) > words_limit:
            # divide text into chunks of max 300 words while preserving sentence boundaries
            chunks = chunk_text_by_word_limit(text, word_limit=words_limit)
            for chunk in chunks:
                block_of_text.append({
                    "id": id,
                    "text": chunk,
                })
        else:
            block_of_text.append({
                "id": id,
                "text": text,
            })
    return block_of_text


In [ ]:
import pandas as pd
df = pd.read_csv('../../../DATA/europeans-news/ner_data_with_article_ids.csv')
df = df[df['filename'] == filename.replace(':', '_')]
blocks = create_block_of_text(df)

In [ ]:
df.shape

In [ ]:
with open(f"{file_name}.txt", "w") as f:
    f.close()

for block in blocks:
    print(f"ID: {block['id']}, Text: {block['text']}")
    entities = model.predict_entities(block["text"], labels)

    # print(entities[0])
    for entity in entities:
        print(entity["text"], "=>", entity["label"])
    print("\n\n")

    tokens = block["text"].split(" ")
    token_offsets = [ (entity["start"], entity["end"]) for entity in entities]

    # print(f"Tokens: {tokens}")
    # print(f"Token count: {len(tokens)}")
    # print(f"Entities: {entities}")
    
    with open(f"{file_name}.txt", "a+") as f:
        token_labels = spans_to_bio(tokens, entities)
        
        for token_info in token_labels.values():
            f.write(f"{token_info['token']}\tPOS\t{token_info['label']}\n")

### Evalaution

In [ ]:
import pandas as pd
true_path = '../../../DATA/europeans-news/bio/'
pred_path = "."

true_df = pd.read_csv(
    os.path.join(true_path, f"{file_name}".replace(':', '_')+'.alto.xml.bio'), 
    sep=' ', 
    header=None, 
    on_bad_lines='skip', 
    engine='python',
    quotechar=None,
    quoting=3
)
pred_df = pd.read_csv(
    os.path.join(pred_path, f"{file_name}.txt"), 
    sep='\t', 
    header=None, 
    on_bad_lines='skip', 
    engine='python',
    quotechar=None,
    quoting=3
    )

true_tags = [tag for tag in true_df.iloc[:, 2]]
pred_tags = [tag for tag in pred_df.iloc[:, 2]]

In [ ]:
from seqeval.metrics import classification_report, f1_score


if len(true_tags) != len(pred_tags):
    print(f"Warning: true tags and predicted tags have different lengths ({len(true_tags)} vs {len(pred_tags)}). Truncating to the shorter length.")
    min_len = min(len(true_tags), len(pred_tags))
    true_tags = true_tags[:min_len]
    pred_tags = pred_tags[:min_len]

print(classification_report([true_tags], [pred_tags]))
print("\nF1 Score (micro avg):", f1_score([true_tags], [pred_tags]))

# Also print as dict for detailed metrics
report_dict = classification_report([true_tags], [pred_tags], output_dict=True)
print("\n" + "=" * 50)
print("DETAILED METRICS")
print("=" * 50)
for entity_type in sorted([k for k in report_dict.keys() if k not in ['micro avg', 'macro avg', 'weighted avg']]):
    metrics = report_dict[entity_type]
    print(f"{entity_type}:")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    print(f"  F1-Score: {metrics['f1-score']:.4f}")

In [ ]:
for idx, (t, p, true_token, pred_token) in enumerate(zip(true_tags, pred_tags, true_df.iloc[:, 0], pred_df.iloc[:, 0])):
    msg = f"True: {t}, Pred: {p}, {true_token}, {pred_token}"
    # if true_token != pred_token:
    #     print(f"STOP! (Index: {idx})")
    #     break
    print(f"True: {t}, Pred: {p}, {true_token}, {pred_token}")

In [ ]:
pred_df.iloc[50]